# Granularity
CodeGraphene supports parsing code at different levels of detail: `LINE`, `METHOD`, and `FILE`.
We will point our pipeline at `sample_code.py` and see how changing the granularity alters the graph.

## Setup and Imports

In [1]:
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Line-Level Granularity
Each node represents a single line. We target the query execution (line 63) and look 1 hop away.

In [2]:
pipeline_line = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.LINE),
    trimmer=KHopTrimmer(hops=1),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE)
)

line_result = pipeline_line.run(target_file, target=63)
print("\n--- FINAL LINE PROMPT ---")
print(line_result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpq8x5fl8g/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpq8x5fl8g/cpg.bin --repr all --out /tmp/tmpq8x5fl8g/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['25769803796', '30064771187', '30064771188', '30064771189', '30064771190', '55834574893', '55834574894', '68719476837', '68719476838', '68719476839', '68719476840', '94489280568'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 12 targets matched; running KHopTrimmer once per target and unioning results.


[Pipeline] Step 3: running CodeReconstructionSerializer...

--- FINAL LINE PROMPT ---
Line 52: self
Line 59: self.cur_batch
Line 62: tmp7 = self.scheduler
self.scheduler.step()
Line 63: tmp7 = self.scheduler
self.scheduler.step()
Line 66: tmp9
Line 67: self
Line 71: self


## Method-Level Granularity
All AST tokens inside a function are collapsed into a single "Method" node. Instead of targeting a line number, we can now target the method by its name!

In [3]:
# With METHOD granularity, the target node must be a string (method name), unlike the line number we passed for LINE granularity.
target_method_name = "step"

pipeline_method = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.METHOD),
    trimmer=KHopTrimmer(hops=2),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.METHOD)
)

method_result = pipeline_method.run(target_file, target=target_method_name)
print("\n--- FINAL METHOD PROMPT ---")
print(method_result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpi2shggow/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpi2shggow/cpg.bin --repr all --out /tmp/tmpi2shggow/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['107374182403', '158913789955', '167503724547'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 3 targets matched; running KHopTrimmer once per target and unioning results.
[Pipeline] Step 3: running CodeReconstructionSerializer...

--- FINAL METHOD PROMPT ---
:<module>.state_dict
:<module>._is_default_fp16
:<module>.configure_device
:<module>.log
:<module>.get_opt_state_for_param
:<module>.configure_distributed_training
:<module>.step_normal
:<module>.get_opt_param_group_for_param
:<module>.zero_grad
:<module>.epoch_callback_exec
:<global>
:<module>.load_state_dict
:<module>.clip_grad
:<module>.recover_states
:<module>.step_after_roll_back
:<module>.load_state_dict
:<module>.gradient_accumulation_boundary
:<module>.step
:<module>.get_batch_single_loader
:<module>.configure_distributed_training
:<module>.epoch_callback_exec
:<module>.get_loss
:<module>.backward
:<module>.state_dict
:<module>.clip_grad
:<module>.get_opt_param_group_for_param


## File-Level Granularity
The entire file is collapsed into one node. In a multi-file repository, edges would connect to imported files.

**Known issue (tracked as [#9](https://github.com/stg-tud/CodeGraphene/issues/9)):** `NodeGranularity.FILE`'s node filter only checks for a `NAME` attribute, but Joern's CPG puts `NAME` on many node types (`METHOD`, `TYPE_DECL`, `IDENTIFIER`, ...), not just `FILE` nodes -- it never checks the node's actual CPG type. The output below is **not** a meaningful file-level summary; it's whatever named nodes happened to survive that overly-permissive filter. Treat this section as a demonstration of the bug, not of a working feature, until #9 is fixed. `LINE` and `METHOD` granularity above are not affected.

In [4]:
# With FILE granularity, the target node must be a string (path to file relative to target_file).
# In this single-file example, we pass the empty string as the target - it points to target_file.

target_name = ""

pipeline_file = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.FILE),
    trimmer=KHopTrimmer(hops=2),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.FILE)
)

file_result = pipeline_file.run(target_file, target=target_name)
print("\n--- FINAL FILE PROMPT ---")
print(file_result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmptgpiremr/cpg.bin


[JoernParser] Running: joern-export /tmp/tmptgpiremr/cpg.bin --repr all --out /tmp/tmptgpiremr/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['21474836480', '21474836481', '21474836482', '21474836483', '21474836484', '21474836485', '21474836486', '21474836487', '21474836488', '21474836489', '21474836490', '21474836491', '21474836492', '21474836493', '21474836494', '21474836495', '21474836496', '21474836497', '21474836498', '21474836499', '21474836500', '21474836501', '21474836502', '21474836503', '21474836504', '21474836505', '21474836506', '21474836507', '30064771469', '30064771613', '30064771617', '60129542144'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 32 targets matched; running KHopTrimmer once per target and unioning results.


[Pipeline] Step 3: running CodeReconstructionSerializer...

--- FINAL FILE PROMPT ---
self
tmp6
step_normal
tmp3
<operator>.multiplication
<operator>.fieldAccess
<operator>.assignmentPlus
self
tmp6
problem
<operator>.assignment
<operator>.fieldAccess
<operator>.assignment
self
<operator>.fieldAccess
<operator>.assignment
<operator>.listLiteral
<module>
self
<operator>.fieldAccess
<operator>.modulo
tmp0
problem
<operator>.assignment
self
<operator>.assignment
self
<operator>.fieldAccess
tmp1
tmp2
problem
global_step
_
tmp4
<operator>.fieldAccess
log
self
tmp2
<operator>.fieldAccess
step
self
step_normal
self
<operator>.fieldAccess
tmp4
<operator>.fieldAccess
self
<operator>.fieldAccess
global_step
<operator>.fieldAccess
<operator>.assignment
self
tmp5
<operator>.fieldAccess
self
<operator>.indexAccess
<operator>.fieldAccess
self
<operator>.fieldAccess
self
__iter__
problem
self
<operator>.equals
self
<operator>.greaterThan
tmp0
step_normal
index
loss_dict
self
self
<operator>.fieldAcces